# Multinomial Classification

**Topics:** Multi-class Classification, Multinomial Logit, Softmax

## Overview

This notebook demonstrates multi-class classification using multinomial logistic regression. We'll classify observations into 3+ categories and interpret class probabilities.

## What You'll Learn

- Fit multinomial GLM for multi-class outcomes
- Interpret coefficients for multiple classes
- Compute multi-class confusion matrices
- Evaluate with macro/micro averaged metrics
- Visualize class probabilities

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax

from aurora.models import fit_glm
from sklearn.metrics import classification_report, confusion_matrix as sklearn_cm

sns.set_style('whitegrid')
np.random.seed(42)

## Generate Multi-Class Data

Customer satisfaction: Low (0), Medium (1), High (2)

In [ ]:
n = 400

# Predictors
price = np.random.uniform(10, 100, n)
quality = np.random.uniform(1, 10, n)
support = np.random.uniform(1, 10, n)

# Linear predictors for each class (Low is reference)
# Medium vs Low
eta_medium = -2 - 0.02*price + 0.3*quality + 0.2*support + np.random.randn(n)*0.5
# High vs Low  
eta_high = -4 - 0.03*price + 0.5*quality + 0.4*support + np.random.randn(n)*0.5

# Compute probabilities via softmax
eta_matrix = np.column_stack([np.zeros(n), eta_medium, eta_high])  # Low is reference (0)
probs = softmax(eta_matrix, axis=1)

# Sample classes
satisfaction = np.array([np.random.choice(3, p=probs[i]) for i in range(n)])

df = pd.DataFrame({
    'price': price,
    'quality': quality,
    'support': support,
    'satisfaction': satisfaction,
    'satisfaction_label': [['Low', 'Medium', 'High'][s] for s in satisfaction]
})

print(f"Generated {n} customers")
print(f"\nClass distribution:")
print(df['satisfaction_label'].value_counts().sort_index())
print(f"\nMean values by satisfaction:")
print(df.groupby('satisfaction_label')[['price', 'quality', 'support']].mean())


## Fit Multinomial Model

For simplicity, we'll use one-vs-rest approach (fit binary models for each class):

In [ ]:
# Design matrix
X = np.column_stack([
    np.ones(n),
    df['price'],
    df['quality'],
    df['support']
])

# Fit separate binary models for each class vs rest
models = {}
for class_idx in range(3):
    y_binary = (df['satisfaction'] == class_idx).astype(int)
    models[class_idx] = fit_glm(X=X, y=y_binary, family='binomial')
    print(f"\nClass {class_idx} ({'Low' if class_idx==0 else 'Medium' if class_idx==1 else 'High'}):")
    print(f"  Converged: {models[class_idx].converged_}")
    print(f"  Deviance: {models[class_idx].deviance_:.2f}")

# Predict probabilities for all classes
probs_pred = np.column_stack([models[i].predict(X) for i in range(3)])
# Normalize to sum to 1
probs_pred = probs_pred / probs_pred.sum(axis=1, keepdims=True)

# Predicted classes
y_pred = probs_pred.argmax(axis=1)

df['satisfaction_pred'] = y_pred
df['satisfaction_pred_label'] = [['Low', 'Medium', 'High'][y] for y in y_pred]


## Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = sklearn_cm(df['satisfaction'], df['satisfaction_pred'])

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'],
            ax=ax)
ax.set_ylabel('True Class')
ax.set_xlabel('Predicted Class')
ax.set_title('Confusion Matrix: Customer Satisfaction')
plt.tight_layout()
plt.show()

# Accuracy
accuracy = (df['satisfaction'] == df['satisfaction_pred']).mean()
print(f"\nOverall Accuracy: {accuracy:.3f}")

## Classification Report

In [ ]:
# Detailed metrics per class
report = classification_report(df['satisfaction'], df['satisfaction_pred'],
                               target_names=['Low', 'Medium', 'High'])
print("\nClassification Report:")
print(report)

## Visualize Predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Quality vs Support colored by true class
for idx, label in enumerate(['Low', 'Medium', 'High']):
    mask = df['satisfaction'] == idx
    axes[0].scatter(df[mask]['quality'], df[mask]['support'],
                   label=label, alpha=0.6, s=50, edgecolor='k', linewidth=0.5)
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Support Score')
axes[0].set_title('True Classes')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Quality vs Support colored by predicted class
for idx, label in enumerate(['Low', 'Medium', 'High']):
    mask = df['satisfaction_pred'] == idx
    axes[1].scatter(df[mask]['quality'], df[mask]['support'],
                   label=label, alpha=0.6, s=50, edgecolor='k', linewidth=0.5)
axes[1].set_xlabel('Quality Score')
axes[1].set_ylabel('Support Score')
axes[1].set_title('Predicted Classes')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Next:** See `02_classification/03_gam_classification.ipynb` for non-linear boundaries